# Retention Gap Aggregate v1

Notebook-first diagnostic for the active objective: keep only a model/reference path that can satisfy the retained-model gate: all required walk-forward windows present, worst walk-forward MaxDD below 20%, YTD return aligned with the current best model, and SOXLBacktest converted qlib buy/sell reference retained as a scoring baseline.

Governance:
- Baselines: `soxlbacktest_qlib_walkforward_baseline_v1`, `model_retention_walkforward_gate_v1`, and current production composer `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: SOXLBacktest qlib buy/sell reference, QuantBench current ensemble screen, state-objective budget-usage BC official replay, teacher risk-budget feasibility screen.
- Role: diagnose / model-selection aggregation, not promotion.
- Changed layer: model selection and training objective routing.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, plus YTD 2026 through the safe replay end.
- Acceptance criteria: complete walk-forward evidence, worst WF MaxDD `< 20%`, YTD return `>= 132.08%` SOXLBacktest reference and `>= 213.14%` current-best alignment.
- Failure mode: high-return paths still violate WF drawdown; low-drawdown paths underparticipate and fail YTD alignment; missing windows block promotion.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists():
            return path
    raise RuntimeError(f'could not locate repo root from {start}')

ROOT = find_repo_root(Path.cwd())
OUT = ROOT / "runtime/model_selection/card/retention_gap_aggregate_v1"
OUT.mkdir(parents=True, exist_ok=True)

REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
MAX_WF_DD = 20.0
SOXLBACKTEST_RETURN = 132.077978
CURRENT_BEST_RETURN = 224.361539
REQUIRED_BEST_RETURN = 213.143462

TARGETS = {
    "max_walkforward_drawdown_pct": MAX_WF_DD,
    "required_soxlbacktest_return_pct": SOXLBACKTEST_RETURN,
    "current_best_return_pct": CURRENT_BEST_RETURN,
    "required_best_return_pct": REQUIRED_BEST_RETURN,
    "required_wf_windows": REQUIRED_WINDOWS,
}

def read_csv(rel: str) -> pd.DataFrame:
    path = ROOT / rel
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)

def read_json(rel: str) -> dict:
    path = ROOT / rel
    if not path.exists():
        return {}
    with path.open() as fh:
        return json.load(fh)

def boolish(value) -> bool:
    if isinstance(value, bool):
        return value
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "pass"}

def numeric(value):
    try:
        if value is None:
            return np.nan
        return float(value)
    except Exception:
        return np.nan

def split_windows(value) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    return [item for item in str(value).split(',') if item]

def clean_for_json(value):
    if isinstance(value, dict):
        return {key: clean_for_json(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clean_for_json(item) for item in value]
    if isinstance(value, tuple):
        return [clean_for_json(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value

print(json.dumps(TARGETS, indent=2))

{
  "max_walkforward_drawdown_pct": 20.0,
  "required_soxlbacktest_return_pct": 132.077978,
  "current_best_return_pct": 224.361539,
  "required_best_return_pct": 213.143462,
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ]
}


In [2]:
summary_rows: list[dict] = []
window_rows: list[dict] = []
source_audit: list[dict] = []

def add_source_audit(source_label: str, path: str, rows: int, note: str = ""):
    source_audit.append({"source_label": source_label, "path": path, "rows": int(rows), "note": note})

def add_summary(source_label: str, candidate: str, row: dict, source_path: str, role: str = "diagnostic"):
    wf_windows = split_windows(row.get('wf_windows_present'))
    if not wf_windows:
        wf_windows = [w for w in REQUIRED_WINDOWS if pd.notna(row.get(f'{w}_max_dd_pct'))]
    complete_wf = all(w in wf_windows for w in REQUIRED_WINDOWS) or boolish(row.get('complete_wf'))
    ytd = numeric(row.get('ytd_return_pct'))
    ytd_dd = numeric(row.get('ytd_max_drawdown_pct'))
    worst_dd = numeric(row.get('worst_wf_max_drawdown_pct'))
    if math.isnan(worst_dd):
        dd_values = [numeric(row.get(f'{w}_max_dd_pct')) for w in REQUIRED_WINDOWS]
        dd_values = [v for v in dd_values if not math.isnan(v)]
        worst_dd = max(dd_values) if dd_values else np.nan
    min_wf_return = numeric(row.get('min_wf_return_pct'))
    mean_wf_return = numeric(row.get('mean_wf_return_pct'))

    wf_dd_pass = bool(complete_wf and not math.isnan(worst_dd) and worst_dd < MAX_WF_DD)
    soxl_ref_pass = bool(not math.isnan(ytd) and ytd >= SOXLBACKTEST_RETURN)
    best_pass = bool(not math.isnan(ytd) and ytd >= REQUIRED_BEST_RETURN)
    full_gate = bool(complete_wf and wf_dd_pass and soxl_ref_pass and best_pass)

    missing_wf = [w for w in REQUIRED_WINDOWS if w not in wf_windows]
    if isinstance(row.get('missing_wf_windows'), str) and row.get('missing_wf_windows'):
        missing_wf = [w for w in str(row.get('missing_wf_windows')).split(',') if w]

    drawdown_gap = max(0.0, worst_dd - MAX_WF_DD) if not math.isnan(worst_dd) else np.nan
    ytd_best_gap = max(0.0, REQUIRED_BEST_RETURN - ytd) if not math.isnan(ytd) else np.nan
    ytd_ref_gap = max(0.0, SOXLBACKTEST_RETURN - ytd) if not math.isnan(ytd) else np.nan
    missing_loss = 100.0 * len(missing_wf) + (75.0 if math.isnan(ytd) else 0.0)
    objective_loss = (0 if math.isnan(ytd_best_gap) else ytd_best_gap) + (0.25 * (0 if math.isnan(ytd_ref_gap) else ytd_ref_gap)) + (10.0 * (0 if math.isnan(drawdown_gap) else drawdown_gap)) + missing_loss

    failures: list[str] = []
    if missing_wf:
        failures += [f'missing_wf_window:{w}' for w in missing_wf]
    if complete_wf and not wf_dd_pass:
        failures.append('worst_wf_max_drawdown_ge_20')
    if not soxl_ref_pass:
        failures.append('ytd_return_below_soxlbacktest_reference' if not math.isnan(ytd) else 'missing_ytd_return')
    if not best_pass:
        failures.append('ytd_return_below_current_best_alignment' if not math.isnan(ytd) else 'missing_ytd_return')
    if not failures and not full_gate:
        failures.append('unclassified_gate_failure')

    if missing_wf:
        bottleneck = 'missing_walkforward_evidence'
    elif not wf_dd_pass and not best_pass:
        bottleneck = 'joint_drawdown_and_return_gap'
    elif not wf_dd_pass:
        bottleneck = 'walkforward_drawdown_gap'
    elif not best_pass:
        bottleneck = 'trend_participation_or_ytd_return_gap'
    else:
        bottleneck = 'gate_pass_candidate' if full_gate else 'unclassified'

    summary_rows.append({
        'candidate_id': f'{source_label}::{candidate}',
        'source_label': source_label,
        'candidate': candidate,
        'role': role,
        'source_path': source_path,
        'complete_wf': complete_wf,
        'wf_windows_present': ','.join(wf_windows),
        'missing_wf_windows': ','.join(missing_wf),
        'worst_wf_max_drawdown_pct': worst_dd,
        'drawdown_gap_pct': drawdown_gap,
        'min_wf_return_pct': min_wf_return,
        'mean_wf_return_pct': mean_wf_return,
        'ytd_return_pct': ytd,
        'ytd_max_drawdown_pct': ytd_dd,
        'ytd_best_gap_pct': ytd_best_gap,
        'ytd_soxlbacktest_gap_pct': ytd_ref_gap,
        'wf_drawdown_pass': wf_dd_pass,
        'ytd_soxlbacktest_return_pass': soxl_ref_pass,
        'ytd_current_best_alignment_pass': best_pass,
        'retention_gate_pass': full_gate,
        'objective_loss': objective_loss,
        'bottleneck': bottleneck,
        'failures': ';'.join(failures),
    })

def add_summary_table(source_label: str, rel: str, candidate_col: str, role: str = 'diagnostic'):
    df = read_csv(rel)
    add_source_audit(source_label, rel, len(df), 'summary table')
    if df.empty:
        return
    for _, row in df.iterrows():
        candidate = str(row.get(candidate_col))
        add_summary(source_label, candidate, row.to_dict(), rel, role=role)

add_summary_table('existing_model_retention_gate', 'runtime/model_selection/card/model_retention_walkforward_gate_v1/retention_gate.csv', 'profile', role='existing_model')
add_summary_table('soxlbacktest_reference_score', 'runtime/model_selection/card/soxlbacktest_reference_model_score_v1/candidate_score_summary.csv', 'profile', role='existing_model_scored_against_reference')
add_summary_table('soxlbacktest_qlib_reference', 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_gate_summary.csv', 'candidate', role='reference_baseline')
add_summary_table('quantbench_current_walkforward', 'runtime/model_selection/card/quantbench_ensemble_current_walkforward_v1/retention_style_gate.csv', 'profile', role='training_target_diagnostic')
add_summary_table('state_conditioned_overlay_gap', 'runtime/model_selection/card/state_conditioned_overlay_gap_to_20_v1/overlay_gate_summary.csv', 'variant', role='fixed_actor_overlay_diagnostic')

len(summary_rows), len(source_audit)

(53, 5)

In [3]:
def summarize_window_metrics(source_label: str, rel: str, candidate: str, window_col: str, return_col: str, buyhold_col: str | None, rel_col: str | None, dd_col: str, avg_pos_col: str | None = None, role: str = 'diagnostic'):
    df = read_csv(rel)
    add_source_audit(source_label, rel, len(df), 'window metrics')
    if df.empty:
        return
    windows = []
    max_dds = []
    returns = []
    avg_positions = []
    for _, row in df.iterrows():
        window = str(row[window_col])
        if window not in REQUIRED_WINDOWS:
            continue
        ret = numeric(row.get(return_col))
        bnh = numeric(row.get(buyhold_col)) if buyhold_col else np.nan
        relret = numeric(row.get(rel_col)) if rel_col else (ret - bnh if not math.isnan(ret) and not math.isnan(bnh) else np.nan)
        dd = numeric(row.get(dd_col))
        avg_pos = numeric(row.get(avg_pos_col)) if avg_pos_col else np.nan
        windows.append(window)
        if not math.isnan(dd): max_dds.append(dd)
        if not math.isnan(ret): returns.append(ret)
        if not math.isnan(avg_pos): avg_positions.append(avg_pos)
        window_rows.append({
            'source_label': source_label, 'candidate': candidate, 'window': window,
            'return_pct': ret, 'buyhold_return_pct': bnh, 'relative_return_pct': relret,
            'max_dd_pct': dd, 'avg_position': avg_pos, 'source_path': rel,
        })
    if windows:
        add_summary(source_label, candidate, {
            'wf_windows_present': ','.join(sorted(set(windows))),
            'worst_wf_max_drawdown_pct': max(max_dds) if max_dds else np.nan,
            'min_wf_return_pct': min(returns) if returns else np.nan,
            'mean_wf_return_pct': float(np.mean(returns)) if returns else np.nan,
            'ytd_return_pct': np.nan,
            'ytd_max_drawdown_pct': np.nan,
        }, rel, role=role)

summarize_window_metrics(
    'teacher_risk_budget_selected',
    'runtime/model_selection/card/soxlbacktest_teacher_risk_budget_feasibility_v1/selected_window_metrics.csv',
    'combined_f00_b416', 'window', 'return', 'buyhold_return', 'relative_return', 'max_dd', 'avg_pos', role='teacher_budget_diagnostic'
)
summarize_window_metrics(
    'soxlbacktest_legacy_teacher',
    'runtime/model_selection/card/soxlbacktest_teacher_risk_budget_feasibility_v1/legacy_teacher_window_metrics.csv',
    'legacy_teacher', 'window', 'return', 'buyhold_return', 'relative_return', 'max_dd', 'avg_pos', role='legacy_teacher_window_path'
)

# Official replay metrics expose candidate/fold rows rather than full summary rows. Add one summary per candidate.
official_rel = 'runtime/model_selection/card/official_replay_risk_state_target_screen_v1/official_replay_metrics.csv'
official = read_csv(official_rel)
add_source_audit('official_risk_state_target_screen', official_rel, len(official), 'candidate-fold official warm replay')
if not official.empty:
    for candidate, group in official.groupby('candidate'):
        row = {
            'wf_windows_present': ','.join(sorted(set(group['fold'].astype(str)))),
            'worst_wf_max_drawdown_pct': group['official_warm_max_dd'].max(),
            'min_wf_return_pct': group['official_warm_return'].min(),
            'mean_wf_return_pct': group['official_warm_return'].mean(),
            'ytd_return_pct': np.nan,
            'ytd_max_drawdown_pct': np.nan,
        }
        add_summary('official_risk_state_target_screen', str(candidate), row, official_rel, role='official_warm_replay_partial')
        for _, r in group.iterrows():
            window_rows.append({
                'source_label': 'official_risk_state_target_screen',
                'candidate': str(candidate),
                'window': str(r['fold']),
                'return_pct': numeric(r.get('official_warm_return')),
                'buyhold_return_pct': numeric(r.get('official_warm_buyhold_return')),
                'relative_return_pct': numeric(r.get('official_warm_relative_return')),
                'max_dd_pct': numeric(r.get('official_warm_max_dd')),
                'avg_position': numeric(r.get('official_warm_avg_pos')),
                'source_path': official_rel,
            })

# Budget usage BC official replay has full fold paths plus YTD gate rows.
budget_rel = 'runtime/model_selection/card/state_objective_budget_usage_bc_official_replay_v1/official_replay_metrics.csv'
budget = read_csv(budget_rel)
add_source_audit('budget_usage_bc_official_replay', budget_rel, len(budget), 'full official replay metrics')
if not budget.empty:
    wf = budget[budget['window'].isin(REQUIRED_WINDOWS)]
    ytd = budget[budget['window'].eq('ytd_2026')]
    add_summary('budget_usage_bc_official_replay', 'risk_budget_usage_v1_state_objective_bc', {
        'wf_windows_present': ','.join(wf['window'].astype(str).tolist()),
        'worst_wf_max_drawdown_pct': wf['max_dd_pct'].max(),
        'min_wf_return_pct': wf['return_pct'].min(),
        'mean_wf_return_pct': wf['return_pct'].mean(),
        'ytd_return_pct': ytd['return_pct'].iloc[0] if len(ytd) else np.nan,
        'ytd_max_drawdown_pct': ytd['max_dd_pct'].iloc[0] if len(ytd) else np.nan,
    }, budget_rel, role='trained_actor_official_replay')
    for _, r in wf.iterrows():
        window_rows.append({
            'source_label': 'budget_usage_bc_official_replay',
            'candidate': 'risk_budget_usage_v1_state_objective_bc',
            'window': str(r['window']),
            'return_pct': numeric(r.get('return_pct')),
            'buyhold_return_pct': numeric(r.get('buyhold_return_pct')),
            'relative_return_pct': numeric(r.get('relative_return_pct')),
            'max_dd_pct': numeric(r.get('max_dd_pct')),
            'avg_position': numeric(r.get('avg_position')),
            'source_path': budget_rel,
        })

if not summary_rows:
    raise RuntimeError(f'no candidate rows loaded; root={ROOT}; audit={source_audit}')

summary_df = pd.DataFrame(summary_rows).drop_duplicates(subset=['candidate_id'], keep='first')
window_df = pd.DataFrame(window_rows)
audit_df = pd.DataFrame(source_audit)

summary_df = summary_df.sort_values(['retention_gate_pass', 'objective_loss'], ascending=[False, True]).reset_index(drop=True)
summary_df.head(10)

,candidate_id,source_label,candidate,role,source_path,complete_wf,wf_windows_present,missing_wf_windows,worst_wf_max_drawdown_pct,drawdown_gap_pct,...,ytd_max_drawdown_pct,ytd_best_gap_pct,ytd_soxlbacktest_gap_pct,wf_drawdown_pass,ytd_soxlbacktest_return_pass,ytd_current_best_alignment_pass,retention_gate_pass,objective_loss,bottleneck,failures
0,existing_model_retention_gate::mode_conditione...,existing_model_retention_gate,mode_conditioned_rebound_reward_train_v2,existing_model,runtime/model_selection/card/model_retention_w...,True,"wf_2020,wf_2022,wf_2023_2024",,21.831720,1.831720,...,27.278641,35.569893,0.0,False,True,False,False,53.887093,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
1,soxlbacktest_reference_score::drawdown_rebound...,soxlbacktest_reference_score,drawdown_rebound_reward_train_v1,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktest_refe...,True,"wf_2020,wf_2022,wf_2023_2024",,20.934520,0.934520,...,27.310377,51.435150,0.0,False,True,False,False,60.780350,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
2,existing_model_retention_gate::drawdown_reboun...,existing_model_retention_gate,drawdown_rebound_reward_train_v1,existing_model,runtime/model_selection/card/model_retention_w...,True,"wf_2020,wf_2022,wf_2023_2024",,23.546830,3.546830,...,27.310377,40.310914,0.0,False,True,False,False,75.779214,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
3,teacher_risk_budget_selected::combined_f00_b416,teacher_risk_budget_selected,combined_f00_b416,teacher_budget_diagnostic,runtime/model_selection/card/soxlbacktest_teac...,True,"wf_2020,wf_2022,wf_2023_2024",,20.692910,0.692910,...,NaN,NaN,NaN,False,False,False,False,81.929103,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;missing_ytd_return...
4,existing_model_retention_gate::state_condition...,existing_model_retention_gate,state_conditioned_drawdown_rebound_reward_trai...,existing_model,runtime/model_selection/card/model_retention_w...,True,"wf_2020,wf_2022,wf_2023_2024",,27.608986,7.608986,...,31.390931,16.638408,0.0,False,True,False,False,92.728268,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
5,soxlbacktest_reference_score::smoke,soxlbacktest_reference_score,smoke,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktest_refe...,False,,wf_2020;wf_2022;wf_2023_2024,NaN,NaN,...,30.346650,0.000000,0.0,False,True,True,False,100.000000,missing_walkforward_evidence,missing_wf_window:wf_2020;wf_2022;wf_2023_2024
6,soxlbacktest_reference_score::state_conditione...,soxlbacktest_reference_score,state_conditioned_drawdown_rebound_reward_trai...,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktest_refe...,True,"wf_2020,wf_2022,wf_2023_2024",,29.147985,9.147985,...,31.390931,16.772974,0.0,False,True,False,False,108.252824,joint_drawdown_and_return_gap,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
7,soxlbacktest_reference_score::balanced_macro_v1,soxlbacktest_reference_score,balanced_macro_v1,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktest_refe...,False,,wf_2020;wf_2022;wf_2023_2024,NaN,NaN,...,29.901822,30.281220,0.0,False,True,False,False,130.281220,missing_walkforward_evidence,missing_wf_window:wf_2020;wf_2022;wf_2023_2024...
8,soxlbacktest_reference_score::balanced_rule_re...,soxlbacktest_reference_score,balanced_rule_residual_v19_reg_v22_retest_v1,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktest_refe...,False,,wf_2020;wf_2022;wf_2023_2024,NaN,NaN,...,27.120494,38.191734,0.0,False,True,False,False,138.191734,missing_walkforward_evidence,missing_wf_window:wf_2020;wf_2022;wf_2023_2024...
9,soxlbacktest_reference_score::mode_conditioned...,soxlbacktest_reference_score,mode_conditioned_rebound_reward_train_v2,existing_model_scored_against_reference,runtime/model_selection/card/soxlbacktes

In [4]:
def first_nonempty(df: pd.DataFrame, mask: pd.Series, sort_col: str, ascending=True):
    subset = df[mask].copy()
    if subset.empty:
        return None
    return subset.sort_values(sort_col, ascending=ascending).iloc[0].to_dict()

full_pass = summary_df[summary_df['retention_gate_pass']]
best_overall = summary_df.iloc[0].to_dict() if len(summary_df) else None
best_complete = first_nonempty(summary_df, summary_df['complete_wf'], 'objective_loss')
best_wf_dd_pass = first_nonempty(summary_df, summary_df['wf_drawdown_pass'] & summary_df['ytd_return_pct'].notna(), 'ytd_return_pct', ascending=False)
best_ytd_alignment = first_nonempty(summary_df, summary_df['ytd_current_best_alignment_pass'] & summary_df['complete_wf'], 'worst_wf_max_drawdown_pct')
best_soxl_ref_alignment = first_nonempty(summary_df, summary_df['ytd_soxlbacktest_return_pass'], 'objective_loss')

mechanism_df = (
    summary_df.groupby(['source_label', 'role'], dropna=False)
    .agg(
        candidates=('candidate_id', 'count'),
        full_gate_count=('retention_gate_pass', 'sum'),
        complete_wf_count=('complete_wf', 'sum'),
        wf_dd_pass_count=('wf_drawdown_pass', 'sum'),
        ytd_best_pass_count=('ytd_current_best_alignment_pass', 'sum'),
        best_objective_loss=('objective_loss', 'min'),
        best_ytd_return_pct=('ytd_return_pct', 'max'),
        best_worst_wf_maxdd_pct=('worst_wf_max_drawdown_pct', 'min'),
    )
    .reset_index()
    .sort_values(['full_gate_count', 'best_objective_loss'], ascending=[False, True])
)

failure_counts = (
    summary_df.assign(failure=summary_df['failures'].fillna('').str.split(';'))
    .explode('failure')
    .query("failure != ''")
    .groupby('failure')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

summary_df.to_csv(OUT / 'candidate_gate_summary.csv', index=False)
window_df.to_csv(OUT / 'candidate_window_metrics.csv', index=False)
mechanism_df.to_csv(OUT / 'mechanism_gap_summary.csv', index=False)
failure_counts.to_csv(OUT / 'failure_counts.csv', index=False)
audit_df.to_csv(OUT / 'source_audit.csv', index=False)

print('candidate_rows', len(summary_df))
print('window_rows', len(window_df))
print('full_gate_count', int(summary_df['retention_gate_pass'].sum()))
print('best_overall', best_overall['candidate_id'] if best_overall else None, best_overall['objective_loss'] if best_overall else None)
print('best_wf_dd_pass', best_wf_dd_pass['candidate_id'] if best_wf_dd_pass else None, best_wf_dd_pass['ytd_return_pct'] if best_wf_dd_pass else None)
print('best_ytd_alignment', best_ytd_alignment['candidate_id'] if best_ytd_alignment else None)

candidate_rows 61
window_rows 19
full_gate_count 0
best_overall existing_model_retention_gate::mode_conditioned_rebound_reward_train_v2 53.887093000000014
best_wf_dd_pass soxlbacktest_qlib_reference::wf_pass_ytd18_15_15_02 18.30891
best_ytd_alignment None


In [5]:
handoff_sources = []
for rel in [
    'runtime/model_selection/card/training_objective_handoff_v1/selection.json',
    'runtime/model_selection/card/warm_start_official_failure_handoff_v1/selection.json',
    'runtime/model_selection/card/cap10_training_objective_handoff_v1/selection.json',
]:
    data = read_json(rel)
    if data:
        handoff_sources.append({
            'path': rel,
            'decision': data.get('decision'),
            'next_mechanism': data.get('next_mechanism') or data.get('required_next_step'),
            'promotion_allowed': data.get('promotion_allowed', False),
        })

all_gate_pass = bool(len(full_pass))
if all_gate_pass:
    decision = 'retention_gap_aggregate_found_candidate_requires_replay_review'
    next_mechanism = 'replay_and_package_selected_retention_candidate'
    promotion_allowed = False
else:
    decision = 'retention_gap_aggregate_no_promotion_train_state_conditioned_multi_objective'
    next_mechanism = 'train_state_conditioned_multi_objective_with_ytd_participation_and_drawdown_gate'
    promotion_allowed = False

selection = {
    'experiment': 'retention_gap_aggregate_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'promotion_allowed': promotion_allowed,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'soxlbacktest_qlib_walkforward_baseline_v1',
        'quantbench_ensemble_current_walkforward_v1',
        'state_objective_budget_usage_bc_official_replay_v1',
        'soxlbacktest_teacher_risk_budget_feasibility_v1',
    ],
    'reference_baseline': 'soxlbacktest_policy_reimplementation_v1',
    'target_thresholds': TARGETS,
    'candidate_count': int(len(summary_df)),
    'full_gate_count': int(summary_df['retention_gate_pass'].sum()),
    'best_overall_candidate': best_overall,
    'best_complete_candidate': best_complete,
    'best_wf_drawdown_pass_candidate_by_ytd': best_wf_dd_pass,
    'best_ytd_alignment_candidate_by_wf_drawdown': best_ytd_alignment,
    'best_soxlbacktest_return_alignment_candidate': best_soxl_ref_alignment,
    'handoff_sources': handoff_sources,
    'next_mechanism': next_mechanism,
    'next_training_objective_terms': [
        'walkforward_drawdown_gap_penalty_below_20pct',
        'ytd_current_best_alignment_floor_213pct',
        'soxlbacktest_buysell_return_floor_132pct',
        'state_conditioned_trend_participation_without_wf2022_dd_drift',
        'official_replay_metric_source_required_for_selection',
    ],
    'sidecars': {
        'candidate_gate_summary': 'runtime/model_selection/card/retention_gap_aggregate_v1/candidate_gate_summary.csv',
        'candidate_window_metrics': 'runtime/model_selection/card/retention_gap_aggregate_v1/candidate_window_metrics.csv',
        'mechanism_gap_summary': 'runtime/model_selection/card/retention_gap_aggregate_v1/mechanism_gap_summary.csv',
        'failure_counts': 'runtime/model_selection/card/retention_gap_aggregate_v1/failure_counts.csv',
        'source_audit': 'runtime/model_selection/card/retention_gap_aggregate_v1/source_audit.csv',
        'selection': 'runtime/model_selection/card/retention_gap_aggregate_v1/selection.json',
        'summary': 'runtime/model_selection/card/retention_gap_aggregate_v1/summary.json',
        'conclusion': 'runtime/model_selection/card/retention_gap_aggregate_v1/conclusion.md',
    },
}

summary = {
    'experiment': 'retention_gap_aggregate_v1',
    'created_at': selection['created_at'],
    'decision': decision,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'candidate_count': selection['candidate_count'],
    'full_gate_count': selection['full_gate_count'],
    'target_thresholds': TARGETS,
    'best_overall_candidate_id': best_overall['candidate_id'] if best_overall else None,
    'best_overall_objective_loss': best_overall['objective_loss'] if best_overall else None,
    'best_wf_drawdown_pass_candidate_by_ytd': best_wf_dd_pass['candidate_id'] if best_wf_dd_pass else None,
    'best_wf_drawdown_pass_ytd_return_pct': best_wf_dd_pass['ytd_return_pct'] if best_wf_dd_pass else None,
    'best_ytd_alignment_candidate_by_wf_drawdown': best_ytd_alignment['candidate_id'] if best_ytd_alignment else None,
    'top_failure_counts': failure_counts.head(8).to_dict('records'),
    'next_mechanism': next_mechanism,
    'sidecars': selection['sidecars'],
}

(OUT / 'selection.json').write_text(json.dumps(clean_for_json(selection), indent=2, allow_nan=False), encoding='utf-8')
(OUT / 'summary.json').write_text(json.dumps(clean_for_json(summary), indent=2, allow_nan=False), encoding='utf-8')

summary

{'experiment': 'retention_gap_aggregate_v1',
 'created_at': '2026-06-13T13:51:43.339052+00:00',
 'decision': 'retention_gap_aggregate_no_promotion_train_state_conditioned_multi_objective',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'candidate_count': 61,
 'full_gate_count': 0,
 'target_thresholds': {'max_walkforward_drawdown_pct': 20.0,
  'required_soxlbacktest_return_pct': 132.077978,
  'current_best_return_pct': 224.361539,
  'required_best_return_pct': 213.143462,
  'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']},
 'best_overall_candidate_id': 'existing_model_retention_gate::mode_conditioned_rebound_reward_train_v2',
 'best_overall_objective_loss': 53.887093000000014,
 'best_wf_drawdown_pass_candidate_by_ytd': 'soxlbacktest_qlib_reference::wf_pass_ytd18_15_15_02',
 'best_wf_drawdown_pass_ytd_return_pct': 18.30891,
 'best_ytd_alignment_candidate_by_wf_drawdown': None,
 'top_failure_counts': [{'failure': '

In [6]:
from IPython.display import Markdown, display

def fmt(v, digits=2):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return 'n/a'
    return f'{float(v):.{digits}f}'

conclusion_lines = [
    '# Retention Gap Aggregate v1 Conclusion',
    '',
    f"Decision: `{decision}`.",
    '',
    f"Scored `{len(summary_df)}` candidate/source rows from `{len(audit_df)}` evidence inputs. Full retained-model gate pass count: `{int(summary_df['retention_gate_pass'].sum())}`.",
    '',
    '## Best Current Evidence',
]
if best_overall:
    conclusion_lines.append(f"- Closest aggregate candidate: `{best_overall['candidate_id']}` with objective loss `{fmt(best_overall['objective_loss'])}`; bottleneck `{best_overall['bottleneck']}`; failures `{best_overall['failures']}`.")
if best_wf_dd_pass:
    conclusion_lines.append(f"- Best candidate that satisfies WF drawdown by YTD return: `{best_wf_dd_pass['candidate_id']}`; YTD `{fmt(best_wf_dd_pass['ytd_return_pct'])}%`, worst WF MaxDD `{fmt(best_wf_dd_pass['worst_wf_max_drawdown_pct'])}%`; still fails `{best_wf_dd_pass['failures']}`.")
if best_ytd_alignment:
    conclusion_lines.append(f"- Best current-best-aligned candidate by WF drawdown: `{best_ytd_alignment['candidate_id']}`; YTD `{fmt(best_ytd_alignment['ytd_return_pct'])}%`, worst WF MaxDD `{fmt(best_ytd_alignment['worst_wf_max_drawdown_pct'])}%`; failures `{best_ytd_alignment['failures']}`.")
else:
    conclusion_lines.append('- No complete candidate with usable WF evidence satisfies the current-best YTD alignment gate.')

conclusion_lines += [
    '',
    '## Mechanism Readout',
    '- Low-drawdown paths exist, but they underparticipate and miss both SOXLBacktest and current-best YTD return floors.',
    '- High-YTD existing models exist, but their complete walk-forward evidence still breaches the 20% MaxDD gate or is missing required windows.',
    '- SOXLBacktest qlib buy/sell reference remains useful as a return/timing baseline, but its raw teacher path is not promotable because WF drawdown is too high; risk-capped teacher variants lose too much return.',
    '- QuantBench current ensemble is the clearest low-DD training-target style, but its YTD return is far below the reference floors, so it is a teacher component, not a retained model.',
    '',
    '## Next Action',
    f"- Route the next training branch to `{next_mechanism}`.",
    '- The next actor screen must use official replay metrics as the selection source, include all three WF windows plus YTD through safe replay end, and reject any checkpoint with WF MaxDD >=20% or YTD <213.14%.',
    '- Do not retain or promote any current candidate from this aggregate card.',
    '',
    '## Sidecars',
    '- `candidate_gate_summary.csv`',
    '- `candidate_window_metrics.csv`',
    '- `mechanism_gap_summary.csv`',
    '- `failure_counts.csv`',
    '- `selection.json` / `summary.json`',
]
conclusion = "\n".join(conclusion_lines)
(OUT / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))

# Retention Gap Aggregate v1 Conclusion

Decision: `retention_gap_aggregate_no_promotion_train_state_conditioned_multi_objective`.

Scored `61` candidate/source rows from `9` evidence inputs. Full retained-model gate pass count: `0`.

## Best Current Evidence
- Closest aggregate candidate: `existing_model_retention_gate::mode_conditioned_rebound_reward_train_v2` with objective loss `53.89`; bottleneck `joint_drawdown_and_return_gap`; failures `worst_wf_max_drawdown_ge_20;ytd_return_below_current_best_alignment`.
- Best candidate that satisfies WF drawdown by YTD return: `soxlbacktest_qlib_reference::wf_pass_ytd18_15_15_02`; YTD `18.31%`, worst WF MaxDD `18.51%`; still fails `ytd_return_below_soxlbacktest_reference;ytd_return_below_current_best_alignment`.
- No complete candidate with usable WF evidence satisfies the current-best YTD alignment gate.

## Mechanism Readout
- Low-drawdown paths exist, but they underparticipate and miss both SOXLBacktest and current-best YTD return floors.
- High-YTD existing models exist, but their complete walk-forward evidence still breaches the 20% MaxDD gate or is missing required windows.
- SOXLBacktest qlib buy/sell reference remains useful as a return/timing baseline, but its raw teacher path is not promotable because WF drawdown is too high; risk-capped teacher variants lose too much return.
- QuantBench current ensemble is the clearest low-DD training-target style, but its YTD return is far below the reference floors, so it is a teacher component, not a retained model.

## Next Action
- Route the next training branch to `train_state_conditioned_multi_objective_with_ytd_participation_and_drawdown_gate`.
- The next actor screen must use official replay metrics as the selection source, include all three WF windows plus YTD through safe replay end, and reject any checkpoint with WF MaxDD >=20% or YTD <213.14%.
- Do not retain or promote any current candidate from this aggregate card.

## Sidecars
- `candidate_gate_summary.csv`
- `candidate_window_metrics.csv`
- `mechanism_gap_summary.csv`
- `failure_counts.csv`
- `selection.json` / `summary.json`